<a href="https://colab.research.google.com/github/aruntakhur/NWP/blob/main/WM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 1
-------------------------------------
✔ Understand PushT dataset \
✔ Build CNN Encoder \
✔ Build Dynamics Model \
✔ Build Decoder \
✔ Predict next frame \

# Mount the Google Drive to fetch the DataSet

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Check if the Data Set is available

In [2]:
import os

DATASET_PATH = "/content/drive/MyDrive/DataSet_NLP/WM"
print(os.listdir(DATASET_PATH))

['point_maze.zip', 'point_maze']


#Step 1: Unzip the Dataset

In [3]:
import zipfile
import os

DATASET_DIR = "/content/drive/MyDrive/DataSet_NLP/WM"
ZIP_FILE = os.path.join(DATASET_DIR, "point_maze.zip")
EXTRACT_DIR = os.path.join(DATASET_DIR, "point_maze")

# Extract only if not already extracted
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted to:", EXTRACT_DIR)

Dataset extracted to: /content/drive/MyDrive/DataSet_NLP/WM/point_maze


#Step 2: Explore the Dataset
point_maze/ \
│ \
├── actions.pth \
├── states.pth \
├── seq_len.pth \
│ \
└── obses/ \
     &emsp;&emsp; episode_0000.pth \
     &emsp;&emsp;     episode_0001.pth \
       &emsp;&emsp;   ... \
       &emsp;&emsp;   episode_1999.pth \

In [4]:
import os

for root, dirs, files in os.walk(EXTRACT_DIR):
    print(root)
    for f in files:
        print("    ", f)

/content/drive/MyDrive/DataSet_NLP/WM/point_maze
/content/drive/MyDrive/DataSet_NLP/WM/point_maze/point_maze
     states.pth
     seq_lengths.pth
     actions.pth
/content/drive/MyDrive/DataSet_NLP/WM/point_maze/point_maze/obses
     episode_821.pth
     episode_1102.pth
     episode_030.pth
     episode_392.pth
     episode_1402.pth
     episode_1538.pth
     episode_1009.pth
     episode_1518.pth
     episode_779.pth
     episode_164.pth
     episode_305.pth
     episode_1597.pth
     episode_275.pth
     episode_1418.pth
     episode_761.pth
     episode_1644.pth
     episode_1998.pth
     episode_975.pth
     episode_087.pth
     episode_246.pth
     episode_545.pth
     episode_1105.pth
     episode_701.pth
     episode_1165.pth
     episode_1008.pth
     episode_071.pth
     episode_1723.pth
     episode_1187.pth
     episode_1721.pth
     episode_749.pth
     episode_1270.pth
     episode_757.pth
     episode_1989.pth
     episode_1673.pth
     episode_1161.pth
     episode_1522

There are around 2000 episodes, each stored as a separate .pth file. \
This format is ideal for learning world models because each file represents one complete trajectory.

# Understand One Episode
Before building a world model, we must answer:

What is stored inside one .pth file?

In [5]:
import torch
import os

DATASET = "/content/drive/MyDrive/DataSet_NLP/WM/point_maze/point_maze/obses"

sample_file = os.path.join(DATASET, "episode_001.pth")

episode = torch.load(sample_file, map_location="cpu")

print(type(episode))

<class 'torch.Tensor'>


In [6]:
import torch
import os

DATASET = "/content/drive/MyDrive/DataSet_NLP/WM/point_maze/point_maze"

states = torch.load(os.path.join(DATASET, "states.pth"))
actions = torch.load(os.path.join(DATASET, "actions.pth"))
seq_len = torch.load(os.path.join(DATASET, "seq_lengths.pth"))

print(type(states))
print(type(actions))
print(type(seq_len))

print("States shape :", states.shape)
print("Actions shape:", actions.shape)
print("Seq_len shape:", seq_len.shape)

#print(states[:1])
print(actions[:1])
#print(seq_len[:])

<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
States shape : torch.Size([2000, 100, 4])
Actions shape: torch.Size([2000, 100, 2])
Seq_len shape: torch.Size([2000])
tensor([[[ 1.8569e-01,  6.8853e-01],
         [ 7.1589e-01,  6.9450e-01],
         [ 2.4713e-01, -2.3124e-01],
         [-4.0493e-01, -8.8657e-01],
         [-4.5469e-01, -4.4670e-02],
         [ 6.2434e-01, -4.0046e-02],
         [-2.1443e-01,  6.7216e-01],
         [-3.2521e-01,  2.9634e-01],
         [-2.6352e-01,  9.1431e-01],
         [-7.1930e-01,  7.4017e-01],
         [-5.2784e-02,  6.0182e-01],
         [ 4.0955e-02,  3.5776e-01],
         [ 4.4127e-01,  1.6404e-01],
         [ 7.4746e-02,  5.1723e-01],
         [-7.8818e-01, -5.2799e-02],
         [-6.2734e-01,  4.7384e-01],
         [-5.6690e-01, -7.2956e-01],
         [-3.5172e-01, -7.0065e-01],
         [-5.5536e-01, -2.2702e-01],
         [ 8.0520e-01, -1.0010e-01],
         [ 2.2613e-01,  8.0470e-01],
         [-8.0144e-01,  9.3962e-01],

#  Compute Episode Boundaries

In [7]:
import numpy as np

episode_lengths = seq_len.numpy()

episode_end = np.cumsum(episode_lengths)

episode_start = np.concatenate(([0], episode_end[:-1]))

print("Number of episodes:", len(episode_lengths))

print("First 5 episode starts:", episode_start[:5])

print("First 5 episode ends:", episode_end[:5])

Number of episodes: 2000
First 5 episode starts: [  0 100 200 300 400]
First 5 episode ends: [100 200 300 400 500]


In [8]:
# !pip install robomimic
# !pip install diffusers
# !pip install imageio
# !pip install h5py
# !pip install einops